In [ ]:
!pip install -U ultralytics

In [ ]:
RUN_DEBUG = False

In [ ]:
import os
import json
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/usmanafzaal/strawberry-disease-detection-dataset")
DST = Path("/kaggle/working/strawberry_yolo")

# clean output dir
if DST.exists():
    shutil.rmtree(DST)

for split in ["train", "val", "test"]:
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)

# class mapping
CLASS_NAMES = [
    "Angular Leafspot",
    "Anthracnose Fruit Rot",
    "Blossom Blight",
    "Gray Mold",
    "Leaf Spot",
    "Powdery Mildew Fruit",
    "Powdery Mildew Leaf",
]
class_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}

def polygon_to_yolo_bbox(points, img_w, img_h):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    width = (x_max - x_min) / img_w
    height = (y_max - y_min) / img_h

    return x_center, y_center, width, height

for split in ["train", "val", "test"]:
    split_dir = SRC / split
    json_files = sorted(split_dir.glob("*.json"))

    converted = 0
    skipped = 0

    for json_path in json_files:
        with open(json_path, "r") as f:
            data = json.load(f)

        img_name = data.get("imagePath")
        img_w = data.get("imageWidth")
        img_h = data.get("imageHeight")
        shapes = data.get("shapes", [])

        if not img_name or not img_w or not img_h:
            skipped += 1
            continue

        src_img = split_dir / img_name
        if not src_img.exists():
            skipped += 1
            continue

        label_lines = []

        for shape in shapes:
            label = shape.get("label", "").strip()
            points = shape.get("points", [])

            if label not in class_to_id:
                continue
            if len(points) < 3:
                continue

            x_center, y_center, width, height = polygon_to_yolo_bbox(points, img_w, img_h)
            class_id = class_to_id[label]

            label_lines.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        # copy image
        shutil.copy2(src_img, DST / "images" / split / img_name)

        # write label txt
        txt_name = Path(img_name).stem + ".txt"
        with open(DST / "labels" / split / txt_name, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"{split}: converted={converted}, skipped={skipped}")

# writes data.yaml
yaml_text = f"""path: {DST}
train: images/train
val: images/val
test: images/test

names:
"""
for i, name in enumerate(CLASS_NAMES):
    yaml_text += f"  {i}: {name}\n"

with open(DST / "data.yaml", "w") as f:
    f.write(yaml_text)

print("Created:", DST / "data.yaml")

In [ ]:
# Debugging - verify YOLO label format
if RUN_DEBUG:
    from pathlib import Path
    
    label_files = list((DST / "labels" / "train").glob("*.txt"))[:5]
    for lf in label_files:
        print("\n", lf.name)
        print(lf.read_text())

In [ ]:
# Debugging - verify bounding boxes
if RUN_DEBUG:
    import cv2
    import matplotlib.pyplot as plt
    from pathlib import Path
    
    CLASS_NAMES = {
        0: "Angular Leafspot",
        1: "Anthracnose Fruit Rot",
        2: "Blossom Blight",
        3: "Gray Mold",
        4: "Leaf Spot",
        5: "Powdery Mildew Fruit",
        6: "Powdery Mildew Leaf",
    }
    
    img_dir = Path("/kaggle/working/strawberry_yolo/images/train")
    lbl_dir = Path("/kaggle/working/strawberry_yolo/labels/train")
    
    sample_images = list(img_dir.glob("*.jpg"))[:3]
    
    for img_path in sample_images:
        label_path = lbl_dir / (img_path.stem + ".txt")
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
    
        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
    
                    cls, xc, yc, bw, bh = parts
                    cls = int(cls)
                    xc, yc, bw, bh = map(float, [xc, yc, bw, bh])
    
                    x1 = int((xc - bw / 2) * w)
                    y1 = int((yc - bh / 2) * h)
                    x2 = int((xc + bw / 2) * w)
                    y2 = int((yc + bh / 2) * h)
    
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img, CLASS_NAMES[cls], (x1, max(20, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")
        plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")   # yolo26n.pt, yolo26s.pt, yolo26m.pt
results = model.train(
    data="/kaggle/working/strawberry_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    lr0=0.01,
    project="/kaggle/working/runs",
    name="strawberry_yolo26s_baseline"
)

In [ ]:
metrics = model.val(data="/kaggle/working/strawberry_yolo/data.yaml")
print(metrics)

In [ ]:
model.predict(
    source="/kaggle/working/strawberry_yolo/images/test",
    save=True,
    conf=0.25,
    project="/kaggle/working/runs",
    name="strawberry_preds"
)